In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import torch
import pickle

from sentence_transformers import SentenceTransformer,InputExample,losses,util

from torch.utils.data import DataLoader

from src.metric import *

/tmp/ipykernel_8649/430595576.py:9: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer,InputExample,losses,util


In [3]:
# from google.colab import drive
# drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# path="../data/cleaned/"
path="/content/drive/MyDrive/Project/Resume/CleanedDf/"

In [7]:
with open(path+'train_df.pkl','rb') as f:
    train_df=pickle.load(f)
    
with open(path+'val_df.pkl','rb') as f:
    val_df=pickle.load(f)
        
with open(path+'test_df.pkl','rb') as f:
    test_df=pickle.load(f)
    


In [8]:
device="cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [22]:
bi_encoder= SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2',device=device)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
label_to_score = {0: 0.0, 1: 0.5, 2: 1.0}

In [12]:
bi_train_examples=[
    InputExample(texts=[j,r],label=float(label_to_score[l]))
    for r,j,l in zip(train_df['resume_text'],train_df['job_description_text'],train_df['label'])
]

bi_train_dataloader=DataLoader(bi_train_examples,shuffle=True,batch_size=64)

bi_train_loss=losses.CoSENTLoss(bi_encoder)

In [ ]:
# model_save_path="../models/baseline/bi_encoder"
model_save_path="/content/drive/MyDrive/Project/Resume/Models/Baseline_BiEncoder"
os.makedirs(model_save_path,exist_ok=True)

In [23]:
epochs = 4
best_score = float('-inf')
min_delta=0.01

print("\tTraining Phase")
for epoch in range(1, epochs + 1):
    print(f"Epoch: {epoch}----------")

    bi_encoder.fit(
        train_objectives=[(bi_train_dataloader, bi_train_loss)],
        epochs=1,
        warmup_steps=int(len(bi_train_dataloader) * epochs * 0.1),
        show_progress_bar=True
    )

    val_resume_emb=bi_encoder.encode(
        val_df['resume_text'].tolist(),
        batch_size=64,convert_to_tensor=True,show_progress_bar=False
    )
    val_jd_emb =bi_encoder.encode(
        val_df['job_description_text'].tolist(),
        batch_size=64,convert_to_tensor=True,show_progress_bar=False
    )

    scores = torch.cosine_similarity(val_resume_emb, val_jd_emb).cpu().numpy()

    metrics = model_evaluation(scores, val_df, 'job_description_text')

    print("NDCG:",metrics['ndcg_val'])
    print("MAP:",metrics['map_score'])

    final_score = (0.6*metrics['ndcg_val'] +
                   0.3*metrics['map_score'] +
                   0.1*(metrics['mrr_score'] + metrics['topk_score']))

    if final_score > best_score+min_delta:
        best_score=final_score
        bi_encoder.save(model_save_path)
        
    


	Training Phase
Epoch: 1----------


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


NDCG: 0.6698996724988499
MAP: 0.7657467748477264


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch: 3----------


Step,Training Loss


NDCG: 0.6866553540489377
MAP: 0.7754691737669918
Epoch: 4----------


Step,Training Loss


NDCG: 0.6842477268165772
MAP: 0.775852483469594


In [24]:
bi_encoder= SentenceTransformer(model_save_path,device=device)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [25]:
val_resume_emb=bi_encoder.encode(val_df['resume_text'].tolist(),
                              batch_size=64,convert_to_tensor=True,
                              show_progress_bar=True)
val_jd_emb = bi_encoder.encode(val_df['job_description_text'].tolist(),
                              batch_size=64,convert_to_tensor=True,
                              show_progress_bar=True)

scores = torch.cosine_similarity(val_resume_emb,val_jd_emb).cpu().numpy()

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

In [26]:
metrics=model_evaluation(scores,val_df,'job_description_text')
print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])

Spearman: 0.41566671531884597
Top-3 Accuracy: 0.9047619047619048
NDCG: 0.6813664265075539
MAP: 0.7759837126086241
MRR: 0.7766121031746032


In [27]:
eval_df=val_df.copy()
eval_df['score']=scores
print("\nScore spread within groups:")
spreads = []
for jd, group in eval_df.groupby('job_description_text'):
    if len(group) > 1:
        spreads.append(group['score'].max() - group['score'].min())

print(f"Mean spread: {np.mean(spreads):.3f}")
print(f"% groups with spread < 0.1:"f"{(np.array(spreads) < 0.1).mean():.3f}")


Score spread within groups:
Mean spread: 0.157
% groups with spread < 0.1:0.148


In [29]:
false_neg = eval_df[(eval_df['label'] == 2) & (eval_df['score'] < -0.3)]

print(f"Good Fit resumes scoring below -0.3: {len(false_neg)}")
print("\nSample false_neg resumes:")

i=0
for jd,group in false_neg.groupby('jd_clean'):
    print("-"*100)
    print(f"JD: {jd[:200]}")
    for _,row in group.head(3).iterrows():
        print(f"Index:{row['index']}")
        print(f"Score: {row['score']:.3f}")
        print(f"Resume: {row['resume_text'][:300]}\n")
    i+=1
    if i==3:
        break
        

Good Fit resumes scoring below -0.3: 0

Sample false_neg resumes:


In [32]:
confused = eval_df[(eval_df['label'] == 1) & (eval_df['score'] < 1.2) & (eval_df['score'] > -0.1)]

print(f"confused predictions: {len(confused)}")
print("\nSample confused resumes:")

i=0
for jd,group in confused.groupby('jd_clean'):
    print("-"*100)
    print(f"JD: {jd[:200]}")
    for _,row in group.head(3).iterrows():
        print(f"Index:{row['index']}")
        print(f"Score: {row['score']:.3f}")
        print(f"Resume: {row['resume_text'][:300]}\n")
    i+=1
    if i==3:
        break
        

confused predictions: 285

Sample confused resumes:
----------------------------------------------------------------------------------------------------
JD: a remote contract senior accountant job headquartered in milwaukee wi is available through accelerate professional talent solutions. this role requires candidates to have + years of accounting experie
Index:3410
Score: 0.801
Resume: Career FocusAccomplished and results oriented Investment professional with strong leadership and interpersonal skills who adds energy and value to an organization's quest for excellence.
Summary of SkillsInternet and Microsoft Office - MS Word, MS Power Point, MS Excel, Pivot Tables, Spreadsheets,Ma

Index:3456
Score: 0.819
Resume: SummarySkilled in high profile account management; knowledgeable in team building, oral and written presentation, and interpersonal communications; proven leadership skills in a cross-functional team environment; experienced on social media platforms; skilled in Microsoft Off

In [33]:
test_resume_emb=bi_encoder.encode(test_df['resume_text'].tolist(),
                              batch_size=64,convert_to_tensor=True,
                              show_progress_bar=True)
test_jd_emb = bi_encoder.encode(test_df['job_description_text'].tolist(),
                              batch_size=64,convert_to_tensor=True,
                              show_progress_bar=True)

scores = torch.cosine_similarity(test_resume_emb,test_jd_emb).cpu().numpy()

metrics=model_evaluation(scores,test_df,'job_description_text')

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

In [34]:
print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])

Spearman: 0.3575362192672595
Top-3 Accuracy: 1.0
NDCG: 0.6410700897119637
MAP: 0.7432020443693003
MRR: 0.7894808743169398
